# Hand landmark sequence collection

This notebook implements a data collection pipeline for hand gesture recognition using MediaPipe-based landmark detection. It captures sequences of hand landmarks from a webcam, processes them in real time, and stores them as structured datasets for later use in training machine learning models. The system guides the user through recording multiple labeled action sequences, ensuring consistency via countdowns and fixed-length buffers, while providing live visual feedback of detected hand landmarks.

In [5]:
import os
import pickle
from typing import Literal
import cv2
import time
import json
from landmarkers.inferences import Inference, InferenceSequence
from landmarkers.mp.hands import MPVideoLandmarker, MediapipeHandsMetadata
import numpy as np


with open('../config.json', 'r') as f:
	config = json.load(f)
common_config = config['common']
ACTIONS = common_config['actions']
SEQUENCE_LENGTH = common_config['sequence_length']
DATA_PATH = common_config['data_path']

collect_config = config['collect_data']
N_SEQUENCES = collect_config['n_sequences']
NUM_LANDMARKS = collect_config['num_landmarks']
COUNTDOWN = collect_config['countdown']
MODEL_PATH = collect_config['model_path']
NUM_HANDS = collect_config['num_hands']
WINDOW_WIDTH = collect_config['window_width']
WINDOW_HEIGHT = collect_config['window_height']


def create_empty_hand() -> Inference:
	"""Create an empty hand inference filled with zeros."""
	zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
	return Inference(
		landmarks=zeros,
		world_landmarks=zeros,
		metadata=MediapipeHandsMetadata(category_name="Right", index=0, score=0.0)
	)


def get_hand(inferences, category_name: Literal['Left', 'Right']) -> Inference:
	"""Retrieve a hand inference by category name or return an empty placeholder."""
	if not inferences:
		return create_empty_hand()
	hands = [inf for inf in inferences if inf.metadata.category_name == category_name]
	if hands:
		return hands[0]
	else:
		zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
		return Inference(
			landmarks=zeros,
			world_landmarks=zeros,
			metadata=MediapipeHandsMetadata(category_name=category_name, index=0, score=0.0)
		)


def draw_landmarks(frame, hand_right, hand_left):
    """Draw right and left hand landmarks on the given frame."""
    for lm in hand_right.landmarks.array:
        x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
        cv2.circle(frame, (x, y), 3, (0,255,0), -1)

    for lm in hand_left.landmarks.array:
        x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
        cv2.circle(frame, (x, y), 3, (0,0,255), -1)


def process_frame(landmarker, frame):
    """Run inference on a frame and return both hands along with the timestamp."""
    ts = int(time.time() * 1000)
    inferences = landmarker.infer(frame, ts)
    hand_right = get_hand(inferences, "Right")
    hand_left = get_hand(inferences, "Left")
    return hand_right, hand_left, ts


def wait_for_start(cap, landmarker, window_name, action, seq_idx):
    """Display live feed until the user presses 's' to start or 'q' to cancel."""
    print(f"\nReady to record: {action} sequence {seq_idx}. Press 's' to start.")

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        hand_right, hand_left, _ = process_frame(landmarker, frame)
        draw_landmarks(frame, hand_right, hand_left)

        cv2.putText(frame, f"Press 's' to start {action}{seq_idx}", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
        cv2.imshow(window_name, frame)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('s'):
            return True
        elif key == ord('q'):
            return False


def run_countdown(cap, landmarker, window_name):
    """Run a visual countdown before starting the recording."""
    start_time = time.time()

    while True:
        elapsed = time.time() - start_time
        remaining = COUNTDOWN - int(elapsed)
        if remaining <= 0:
            break

        ret, frame = cap.read()
        if not ret:
            continue

        hand_right, hand_left, _ = process_frame(landmarker, frame)
        draw_landmarks(frame, hand_right, hand_left)

        cv2.putText(frame, f"Starting in {remaining}...", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        cv2.imshow(window_name, frame)
        cv2.waitKey(1)

        if int(elapsed) != int(elapsed - 0.05):
            print(f"{remaining}...")

    print("Recording now!")


def capture_sequence(cap, landmarker, window_name, action, seq_idx, sequence_length):
    """Capture a fixed-length sequence of hand inferences."""
    seq_right = InferenceSequence(fixed_buffer_length=sequence_length)
    seq_left = InferenceSequence(fixed_buffer_length=sequence_length)

    frames_captured = 0

    while frames_captured < sequence_length:
        ret, frame = cap.read()
        if not ret:
            continue

        hand_right, hand_left, ts = process_frame(landmarker, frame)

        seq_right.append(hand_right, ts)
        seq_left.append(hand_left, ts)

        draw_landmarks(frame, hand_right, hand_left)

        cv2.putText(frame,
                    f"{action}{seq_idx} frame {frames_captured+1}/{sequence_length}",
                    (10,80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

        cv2.imshow(window_name, frame)

        frames_captured += 1
        if cv2.waitKey(1) & 0xFF == ord('q'):
            return None, None

    return seq_right, seq_left


def save_sequence(seq_right, seq_left, action, seq_idx):
    """Persist recorded sequences to disk as pickle files."""
    seq_folder = os.path.join(DATA_PATH, action, f"seq_{seq_idx}")
    os.makedirs(seq_folder, exist_ok=True)

    with open(os.path.join(seq_folder, "right.pkl"), "wb") as f:
        pickle.dump(seq_right, f)

    with open(os.path.join(seq_folder, "left.pkl"), "wb") as f:
        pickle.dump(seq_left, f)

    print(f"Sequence saved in {seq_folder}")


def record_sequence(landmarker, cap, action, seq_idx, sequence_length):
    """Handle the full pipeline for recording a single sequence."""
    window_name = f"{action}{seq_idx}"

    ret, frame = cap.read()
    if not ret:
        return False

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, WINDOW_WIDTH, WINDOW_HEIGHT)

    if not wait_for_start(cap, landmarker, window_name, action, seq_idx):
        cv2.destroyWindow(window_name)
        return False

    run_countdown(cap, landmarker, window_name)

    seq_right, seq_left = capture_sequence(
        cap, landmarker, window_name, action, seq_idx, sequence_length
    )

    if seq_right is None:
        cv2.destroyWindow(window_name)
        return False

    save_sequence(seq_right, seq_left, action, seq_idx)

    cv2.destroyWindow(window_name)
    return True


def record_actions(landmarker, cap):
	"""Iterate through all actions and record the configured number of sequences."""
	for action in ACTIONS:
		for seq_idx in range(N_SEQUENCES):
			success = record_sequence(landmarker, cap, action, seq_idx, SEQUENCE_LENGTH)
			if not success:
				print("Recording interrupted by user")
				return


def main():
	"""Initialize resources and start the recording pipeline."""
	os.makedirs(DATA_PATH, exist_ok=True)
	for action in ACTIONS:
		os.makedirs(os.path.join(DATA_PATH, action), exist_ok=True)
	
	cap = cv2.VideoCapture(0)
	try:
		with MPVideoLandmarker(model_path=MODEL_PATH, num_hands=NUM_HANDS) as landmarker:
			print("Starting sequence recording...")
			record_actions(landmarker, cap)
	finally:
		cap.release()
		cv2.destroyAllWindows()
		print("Recording finished")


if __name__ == "__main__":
	main()

W0000 00:00:1775306986.074928  770013 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775306986.087610  770013 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Starting sequence recording...

Ready to record: A sequence 0. Press 's' to start.
1...
1...
Recording now!
Sequence saved in ./dataset/A/seq_0

Ready to record: A sequence 1. Press 's' to start.
1...
Recording now!
Sequence saved in ./dataset/A/seq_1

Ready to record: A sequence 2. Press 's' to start.
Recording interrupted by user
Recording finished
